In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: genera heatmaps de la FAP para cada uno de los indices climaticos 
# Periodo: 2000-2024
# ==========================================

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# =========================
# CONFIGURACIÓN DE RUTAS
# =========================
archivo_entrada = "RUTA DEL ARCHIVO"

carpeta = os.path.dirname(archivo_entrada)
carpeta_salida = os.path.join(carpeta, "NOMBRE DE SALIDA")
os.makedirs(carpeta_salida, exist_ok=True)

# =========================
# PROCESAMIENTO DE DATOS
# =========================
df = pd.read_csv(archivo_entrada)
df["Anio"] = pd.to_numeric(df["Anio"], errors="coerce")
df["FAP_%"] = pd.to_numeric(df["FAP_%"], errors="coerce")

edad_order = ["00_04", "05_14", "15_24", "25_34", "35_44", "45_54", "55_64", "65+"]
edad_labels = ["0-4", "5-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65+"]

indice_order = ["Tx90p", "Tn10p", "Tx10p", "tn90p"]

nombres_indices = {
    "Tn10p": "Noches Frías (Tn10p)", 
    "tn90p": "Noches Cálidas (Tn90p)", 
    "Tx10p": "Días Fríos (Tx10p)", 
    "Tx90p": "Días Cálidos (Tx90p)"
}

colores_indices = {
    "Tn10p": "Blues",
    "Tx10p": "Blues",
    "tn90p": "Reds",
    "Tx90p": "Reds"
}

global_max = max(float(df["FAP_%"].max()), 1)
sns.set_style("white")

# =========================
# GENERACIÓN DE FIGURAS
# =========================
for enfermedad in df["Enfermedad"].dropna().unique():
    
    indices_presentes = [i for i in indice_order if i in df[df["Enfermedad"] == enfermedad]["Indice"].unique()]
    
    if not indices_presentes:
        continue

    fig = plt.figure(figsize=(22, 14)) 
    
    # Espacio entre los bloques grandes (Izquierda y Derecha)
    outer = fig.add_gridspec(2, 2, wspace=0.25, hspace=0.45)

    for k, indice in enumerate(indices_presentes):
        fila, col = k // 2, k % 2
        
        # REDUCCIÓN DEL ESPACIO INTERNO: 
        # Cambiamos wspace de 0.4 a 0.22 para cerrar el hueco punteado
        inner = outer[fila, col].subgridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.22)
        
        ax_h = fig.add_subplot(inner[0, 0])
        ax_m = fig.add_subplot(inner[0, 1])
        cax = fig.add_subplot(inner[0, 2])

        cmap_actual = colores_indices.get(indice, "Reds")

        for i, (ax, sexo_label) in enumerate(zip([ax_h, ax_m], ["Hombres", "Mujeres"])):
            sub_sexo = df[(df["Enfermedad"] == enfermedad) & 
                          (df["Indice"] == indice) & 
                          (df["Sexo"] == sexo_label.upper())].copy()
            
            if sub_sexo.empty:
                continue

            tabla = sub_sexo.pivot_table(
                index="Edad_gpo", columns="Anio", values="FAP_%", aggfunc="mean"
            ).reindex(index=edad_order)

            sns.heatmap(tabla, ax=ax, cmap=cmap_actual, vmin=0, vmax=global_max,
                        linewidths=0.5, linecolor="#f9f9f9", 
                        cbar=True, cbar_ax=cax,
                        cbar_kws={
                            "label": r"$\bf{FAP\ (\%)}$" if i == 1 else ""
                        })

            ax.set_title(sexo_label, fontsize=14, pad=10, fontweight="normal")
            ax.set_xlabel("Año", fontweight="bold", fontsize=11)
            
            # Ajuste de etiquetas de edad
            ax.set_ylabel("Grupo de edad", fontweight="bold", fontsize=11)
            ax.set_yticklabels(edad_labels, rotation=0, fontsize=10)

            years = tabla.columns
            ax.set_xticks(np.arange(len(years)) + 0.5)
            ax.set_xticklabels([str(int(y)) if idx % 2 == 0 else "" for idx, y in enumerate(years)], 
                               rotation=45, ha="right", fontsize=10)

        nombre_display = nombres_indices.get(indice, indice.upper())
        ax_h.annotate(f"{nombre_display}", xy=(1.08, 1.18), xycoords='axes fraction',
                      ha='center', va='center', fontsize=17, fontweight='bold', color="#333333")

    # Títulos generales
    fig.text(0.5, 0.97, "FRACCIÓN ATRIBUIBLE POBLACIONAL", ha='center', fontsize=24, fontweight='bold')
    fig.text(0.5, 0.94, f"{enfermedad.title()}", ha='center', fontsize=20, fontweight='normal')

    plt.subplots_adjust(top=0.88, bottom=0.08, left=0.07, right=0.93)

    nombre_limpio = enfermedad.replace(" ", "_").replace("/", "-")
    plt.savefig(os.path.join(carpeta_salida, f"{nombre_limpio}.png"), dpi=300, bbox_inches="tight")
    plt.close()

print(f"Proceso completado.")

Proceso completado.
